# 1x1-conv-channel-reshape — worked example 1: 1x1 conv that reduces channels matches per-pixel matmul

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `1x1-conv-channel-reshape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A `nn.Conv2d(C_in, C_out, kernel_size=1)` touches one pixel at a time: it linearly combines the `C_in` channel values at each spatial location into `C_out` outputs, with no spatial mixing. When `C_out < C_in` this is a channel reduction (the ResNet bottleneck squeeze). The exact computation is `out[:, :, h, w] = W @ x[:, :, h, w] + b`, where `W` is the conv weight reshaped to `(C_out, C_in)`.

## Worked solution

We want to show that a 1x1 conv that maps `C_in=6` channels down to `C_out=2` is just a per-pixel matrix multiply.

1. **Read the conv weight.** `conv.weight` has shape `(C_out, C_in, 1, 1) = (2, 6, 1, 1)`. The trailing `1, 1` is the (degenerate) spatial kernel. Squeeze it away with `.view(C_out, C_in)` to get the `(2, 6)` matrix `W` that acts on a single pixel's channel vector.

2. **Flatten the spatial grid into a batch of pixels.** Every pixel is independent, so `einops.rearrange(x, 'b c h w -> (b h w) c')` stacks all `B*H*W` pixels as rows, each row a length-`C_in` feature vector.

3. **Apply the linear map.** `x_flat @ W.t()` gives `(B*H*W, C_out)`. Adding `conv.bias` broadcasts the length-`C_out` bias across every row. This is exactly `W @ pixel + b` done for all pixels at once.

4. **Fold the grid back.** `rearrange(... '(b h w) c -> b c h w')` restores the image layout with the new channel count. The spatial dims `H, W` are untouched because a 1x1 conv never mixes neighbors.

The result matches `conv(x)` to fp32 tolerance because it IS the same arithmetic, just reorganized.

In [ ]:
import torch.nn as nn

def one_by_one_channel_reduce(x, conv):
    OC, IC, _, _ = conv.weight.shape
    B, _, H, W = x.shape
    Wlin = conv.weight.view(OC, IC)
    x_flat = rearrange(x, 'b c h w -> (b h w) c')
    out_flat = x_flat @ Wlin.t()
    if conv.bias is not None:
        out_flat = out_flat + conv.bias
    return rearrange(out_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)

t.manual_seed(0)
conv = nn.Conv2d(6, 2, kernel_size=1)
x = t.randn(2, 6, 4, 5)
out = one_by_one_channel_reduce(x, conv)
print('output shape:', tuple(out.shape))
print('matches conv:', t.allclose(out, conv(x), atol=1e-5))